In [1]:
import pandas as pd
from ragas import evaluate
from ragas.dataset_schema import EvaluationDataset, SingleTurnSample
from ragas.metrics import AnswerCorrectness, AnswerRelevancy, ContextPrecision, ContextRecall, Faithfulness
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
import nest_asyncio

/Users/cberches/Documents/GitHub/AttritionTest/capstone_env/lib/python3.9/site-packages/urllib3/__init__.py:34: NotOpenSSLWarning: urllib3 v2.0 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/cberches/Documents/GitHub/AttritionTest/capstone_env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/js/rvl93zld41n502z8rv_gr2qw0000gq/T/ipykernel_44718/603833031.py:4: DeprecationWarning: Importing AnswerCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerCorrectness
  from ragas.metrics import AnswerCorrectness, AnswerRelevancy, ContextPrecision, ContextRecall,

In [2]:
from agent import policy_retriever, rag_prompt

# Initialize LLM and Embeddings for Evaluation
eval_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
eval_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

/Users/cberches/Documents/GitHub/AttritionTest/capstone_env/lib/python3.9/site-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [3]:

# 1. Load your golden test set dataset (CSV with 'question', 'ground_truth')
test_df = pd.read_csv('datasets/hr_testset.csv')

samples = []

for _, row in test_df.iterrows():
    question = row['question']
    ground_truth = row['ground_truth']
    
    # 2. Run the question through the imported RAG retriever logic
    retrieved_docs = policy_retriever.invoke(question)
    contexts = [doc.page_content for doc in retrieved_docs]
    
    response = eval_llm.invoke(rag_prompt.format(context="\n".join(contexts), question=question))
    answer = response.content
    
    # 3. Format as a SingleTurnSample for RAGAS 0.4.3
    samples.append(
        SingleTurnSample(
            user_input=question,
            response=answer,
            retrieved_contexts=contexts,
            reference=ground_truth,
        )
    )

# 4. Create Evaluation Dataset and Run RAGAS Evaluation
eval_dataset = EvaluationDataset(samples=samples)
metrics = [Faithfulness(), AnswerRelevancy(), ContextRecall(), ContextPrecision(), AnswerCorrectness()]

results = evaluate(
    dataset=eval_dataset,
    metrics=metrics,
    llm=eval_llm,
    embeddings=eval_embeddings,
)

# 5. Output the final DataFrame containing the scores
print(results.to_pandas())

Evaluating:   1%|          | 5/500 [00:04<08:19,  1.01s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating:   7%|▋         | 35/500 [01:43<19:41,  2.54s/it]/Users/cberches/Documents/GitHub/AttritionTest/capstone_env/lib/python3.9/site-packages/ragas/metrics/_answer_similarity.py:98: RuntimeWarning: divide by zero encountered in matmul
  similarity = embedding_1_normalized @ embedding_2_normalized.T
/Users/cberches/Documents/GitHub/AttritionTest/capstone_env/lib/python3.9/site-packages/ragas/metrics/_answer_similarity.py:98: RuntimeWarning: overflow encountered in matmul
  similarity = embedding_1_normalized @ embedding_2_normalized.T
/Users/cberches/Documents/GitHub/AttritionTest/capstone_env/lib/python3.9/site-packages/ragas/metrics/_answer_similarity.py:98: RuntimeWarning: inval

                                           user_input retrieved_contexts  \
0   What is the cash bonus for the Bayanihan Loyal...                 []   
1   How many days of paid sabbatical leave are emp...                 []   
2   What are the eligibility requirements for an e...                 []   
3   What is the baseline minimum Percent Salary Hi...                 []   
4   What percentage range of Percent Salary Hike s...                 []   
..                                                ...                ...   
95  What is the percentage weight assigned to Tota...                 []   
96  What is the percentage weight assigned to Perc...                 []   
97  What is the percentage weight assigned to Year...                 []   
98  What is the percentage weight assigned to Trai...                 []   
99  What is the percentage weight assigned to JobR...                 []   

                                             response  \
0   The policy does not cover 

In [8]:
results

{'faithfulness': 0.0000, 'answer_relevancy': 0.3615, 'context_recall': 0.0000, 'context_precision': 0.0000, 'answer_correctness': 0.1486}

In [ ]:

results.to_pandas()

,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_recall,context_precision,answer_correctness
0,What is the cash bonus for the Bayanihan Loyal...,[],The policy does not cover the cash bonus for t...,"Upon reaching 10 Total Working Years, employee...",0.0,0.868974,0.0,0.0,0.178201
1,How many days of paid sabbatical leave are emp...,[],The policy does not cover it.,Employees surpassing 10 Total Working Years ar...,0.0,0.229789,0.0,0.0,0.034483
2,What are the eligibility requirements for an e...,[],The policy does not cover the eligibility requ...,Employees with over 5 Total Working Years auto...,0.0,0.853876,0.0,0.0,0.184692
3,What is the baseline minimum Percent Salary Hi...,[],The policy does not cover the baseline minimum...,All non-PIP employees must receive a baseline ...,0.0,0.775612,0.0,0.0,0.448606
4,What percentage range of Percent Salary Hike s...,[],The policy does not cover the percentage range...,High-flight-risk employees with an Outstanding...,0.0,0.000000,0.0,0.0,0.496821
...,...,...,...,...,...,...,...,...,...
95,What is the percentage weight assigned to Tota...,[],The policy does not cover it.,18.88%.,0.0,0.166937,0.0,0.0,0.031033
96,What is the percentage weight assigned to Perc...,[],The policy does not cover the percentage weigh...,6.53%.,0.0,0.904315,0.0,0.0,0.079895
97,What is the percentage weight assigned to Year...,[],The policy does not cover it.,5.55%.,0.0,0.160620,0.0,0.0,0.039968
98,What is the percentage weight assigned to Trai...,[],The policy does not cover the percentage weigh...,3.28%.,0.0,0.942297,0.0,0.0,0.076255
